# Multi-Site No-Till Field Trial Statistical Analysis

**Author:** Sydney Seiter  
**Purpose:** Demonstration of field trial analysis for multi-site soil health research  
**Skills Demonstrated:**  
- Experimental design and statistical analysis
- Multi-site agricultural data synthesis
- Soil health metrics interpretation
- Publication-quality visualization
- Agronomic recommendation development

---

## Research Context

This notebook analyzes data from a hypothetical multi-site no-till adoption trial similar to studies conducted by the Soil Health Institute. The research question mirrors real-world soil health research:

**Question:** Does no-till management improve soil health indicators compared to conventional tillage across diverse climatic regions?

**Design:** Randomized complete block design (RCBD) with:
- 3 sites (North Carolina, Iowa, Montana)
- 2 treatments (No-till vs. Conventional tillage)
- 4 replicate blocks per site
- 3 years of data (2023-2025)
- Multiple soil health indicators measured

This analysis demonstrates the statistical methods needed to support multi-site research programs and communicate findings to stakeholders.

## Setup: Configure Output Directory

Choose where to save output files. Uncomment the Google Drive option if you want to save to Drive.

In [ ]:
import os

# OPTION 1: Save to current directory (default)
output_dir = '.'

# OPTION 2: Create a dedicated output folder
# output_dir = 'soil_health_outputs'
# os.makedirs(output_dir, exist_ok=True)

# OPTION 3: Save to Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# output_dir = '/content/drive/MyDrive/SoilHealthPortfolio'
# os.makedirs(output_dir, exist_ok=True)

print(f"✓ Outputs will be saved to: {output_dir}")

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Install statsmodels if needed (uncomment for Colab)
# !pip install -q statsmodels

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Set style for publication-quality plots
sns.set_style('whitegrid')
sns.set_context('talk')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

np.random.seed(42)
print("Libraries loaded successfully")

## 1. Generate Realistic Multi-Site Trial Data

Simulating 3 years of soil health data with realistic treatment effects and site variability.

In [ ]:
def generate_trial_data():
    """
    Generate realistic multi-site no-till trial data.
    Treatment effects and site characteristics based on published literature.
    """
    
    sites = ['North Carolina', 'Iowa', 'Montana']
    treatments = ['Conventional Tillage', 'No-Till']
    years = [2023, 2024, 2025]
    blocks = [1, 2, 3, 4]
    
    data_rows = []
    
    # Site-specific baseline characteristics
    site_characteristics = {
        'North Carolina': {
            'base_om': 2.0,  # Lower OM in sandy Coastal Plain soils
            'base_aggregate': 35,  # Lower stability in sandy soils
            'base_bulk_density': 1.55,  # Higher BD in compacted sandy soils
            'base_infiltration': 1.2,  # cm/hr - moderate
            'tillage_effect': 0.8  # Smaller no-till benefit in sandy soils
        },
        'Iowa': {
            'base_om': 4.0,  # Higher OM in Midwest prairie soils
            'base_aggregate': 55,  # Better natural aggregation
            'base_bulk_density': 1.35,  # Lower BD in high-OM soils
            'base_infiltration': 2.0,  # Better infiltration
            'tillage_effect': 1.2  # Larger no-till benefit
        },
        'Montana': {
            'base_om': 3.0,  # Moderate OM in semi-arid region
            'base_aggregate': 45,  # Moderate aggregation
            'base_bulk_density': 1.45,  # Moderate BD
            'base_infiltration': 1.5,  # Moderate infiltration
            'tillage_effect': 1.0  # Moderate no-till benefit
        }
    }
    
    plot_id = 1
    
    for site in sites:
        site_char = site_characteristics[site]
        
        for year in years:
            # Time effect - soil health improves over time with no-till
            year_effect = (year - 2023) * 0.15
            
            for block in blocks:
                # Block effect (field spatial variability)
                block_effect = np.random.normal(0, 0.1)
                
                for treatment in treatments:
                    # Treatment effect
                    if treatment == 'No-Till':
                        treatment_multiplier = site_char['tillage_effect']
                    else:
                        treatment_multiplier = 1.0
                    
                    # Organic matter (%) - increases with no-till over time
                    om_base = site_char['base_om'] * treatment_multiplier
                    om = om_base + (year_effect if treatment == 'No-Till' else 0) + \
                         block_effect + np.random.normal(0, 0.15)
                    
                    # Aggregate stability (%) - improves with no-till
                    agg_base = site_char['base_aggregate'] * treatment_multiplier
                    aggregate_stability = agg_base + (year_effect * 5 if treatment == 'No-Till' else 0) + \
                                         block_effect * 2 + np.random.normal(0, 3)
                    
                    # Bulk density (g/cm³) - decreases with no-till
                    bd_base = site_char['base_bulk_density'] / treatment_multiplier
                    bulk_density = bd_base - (year_effect * 0.02 if treatment == 'No-Till' else 0) + \
                                  block_effect * 0.02 + np.random.normal(0, 0.03)
                    
                    # Infiltration rate (cm/hr) - increases with no-till
                    infil_base = site_char['base_infiltration'] * treatment_multiplier
                    infiltration = infil_base + (year_effect * 0.3 if treatment == 'No-Till' else 0) + \
                                  block_effect * 0.1 + np.random.normal(0, 0.15)
                    
                    # Microbial biomass C (mg/kg) - sensitive indicator, increases with no-till
                    mbc_base = 300 * treatment_multiplier
                    microbial_biomass = mbc_base + (year_effect * 50 if treatment == 'No-Till' else 0) + \
                                       block_effect * 20 + np.random.normal(0, 30)
                    
                    # Crop yield (bu/acre) - economic outcome
                    # No-till may have slight yield penalty initially, benefit later
                    yield_effect = -5 if (treatment == 'No-Till' and year == 2023) else \
                                  (5 * (year - 2023) if treatment == 'No-Till' else 0)
                    base_yield = {'North Carolina': 45, 'Iowa': 180, 'Montana': 50}[site]
                    crop_yield = base_yield + yield_effect + block_effect * 5 + np.random.normal(0, 8)
                    
                    data_rows.append({
                        'plot_id': f'{site[:2]}-{year}-B{block}-{treatment[:4]}',
                        'site': site,
                        'year': year,
                        'block': block,
                        'treatment': treatment,
                        'organic_matter_pct': max(0.5, om),
                        'aggregate_stability_pct': np.clip(aggregate_stability, 10, 95),
                        'bulk_density_g_cm3': np.clip(bulk_density, 1.0, 1.8),
                        'infiltration_cm_hr': max(0.3, infiltration),
                        'microbial_biomass_C_mg_kg': max(100, microbial_biomass),
                        'crop_yield_bu_acre': max(20, crop_yield)
                    })
    
    return pd.DataFrame(data_rows)

# Generate trial data
trial_data = generate_trial_data()

print(f"Trial dataset generated: {len(trial_data)} observations")
print(f"\nExperimental design:")
print(f"  Sites: {trial_data['site'].nunique()}")
print(f"  Years: {trial_data['year'].nunique()}")
print(f"  Treatments: {trial_data['treatment'].nunique()}")
print(f"  Blocks per site: {trial_data['block'].nunique()}")
print(f"\nObservations per site-year-treatment combination: {len(trial_data) // (3*3*2)}")
print("\nFirst 10 rows:")
print(trial_data.head(10))

## 2. Exploratory Data Analysis

Visual exploration of treatment effects and temporal trends.

In [ ]:
# Create comprehensive visualization of soil health indicators
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Soil Health Indicators: No-Till vs. Conventional Tillage Across Sites', 
             fontsize=16, fontweight='bold', y=1.00)

indicators = [
    ('organic_matter_pct', 'Soil Organic Matter (%)', axes[0,0]),
    ('aggregate_stability_pct', 'Aggregate Stability (%)', axes[0,1]),
    ('bulk_density_g_cm3', 'Bulk Density (g/cm³)', axes[0,2]),
    ('infiltration_cm_hr', 'Infiltration Rate (cm/hr)', axes[1,0]),
    ('microbial_biomass_C_mg_kg', 'Microbial Biomass C (mg/kg)', axes[1,1]),
    ('crop_yield_bu_acre', 'Crop Yield (bu/acre)', axes[1,2])
]

for indicator, title, ax in indicators:
    sns.boxplot(data=trial_data, x='site', y=indicator, hue='treatment', ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Site')
    ax.set_ylabel(title)
    ax.legend(title='Treatment', loc='best')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'soil_health_treatment_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Treatment comparison visualizations created")

## 3. Statistical Analysis: Two-Way ANOVA

Testing for treatment and site effects on soil health indicators.

In [ ]:
def perform_anova(data, response_var, title):
    """
    Perform two-way ANOVA with treatment, site, and their interaction.
    Returns ANOVA table and effect sizes.
    """
    # Fit model: response ~ treatment + site + treatment:site
    model = ols(f'{response_var} ~ C(treatment) + C(site) + C(treatment):C(site)', 
                data=data).fit()
    anova_table = anova_lm(model, typ=2)
    
    print(f"\n{'='*70}")
    print(f"ANOVA: {title}")
    print(f"{'='*70}")
    print(anova_table)
    
    # Interpret results
    alpha = 0.05
    print(f"\nInterpretation (α = {alpha}):")
    
    for effect in ['C(treatment)', 'C(site)', 'C(treatment):C(site)']:
        if effect in anova_table.index:
            p_value = anova_table.loc[effect, 'PR(>F)']
            f_value = anova_table.loc[effect, 'F']
            
            effect_name = effect.replace('C(', '').replace(')', '').replace(':', ' × ')
            
            if p_value < alpha:
                print(f"  {effect_name}: SIGNIFICANT (F={f_value:.2f}, p={p_value:.4f})")
            else:
                print(f"  {effect_name}: Not significant (F={f_value:.2f}, p={p_value:.4f})")
    
    # R-squared
    print(f"\nModel R²: {model.rsquared:.3f} (explains {model.rsquared*100:.1f}% of variance)")
    
    return anova_table, model

# Analyze key soil health indicators
anova_results = {}

anova_results['om'] = perform_anova(trial_data, 'organic_matter_pct', 
                                     'Soil Organic Matter')

anova_results['agg'] = perform_anova(trial_data, 'aggregate_stability_pct', 
                                      'Aggregate Stability')

anova_results['bd'] = perform_anova(trial_data, 'bulk_density_g_cm3', 
                                     'Bulk Density')

anova_results['yield'] = perform_anova(trial_data, 'crop_yield_bu_acre', 
                                        'Crop Yield')

## 4. Post-Hoc Analysis: Treatment Means Comparison

Tukey HSD test to identify significant pairwise differences.

In [ ]:
# Create treatment-site combinations for detailed comparison
trial_data['treatment_site'] = trial_data['treatment'] + ' - ' + trial_data['site']

# Tukey HSD for organic matter
print("\n" + "="*70)
print("TUKEY HSD POST-HOC TEST: Soil Organic Matter")
print("="*70)

tukey_om = pairwise_tukeyhsd(endog=trial_data['organic_matter_pct'],
                             groups=trial_data['treatment_site'],
                             alpha=0.05)
print(tukey_om)

# Summary statistics by treatment and site
print("\n" + "="*70)
print("TREATMENT MEANS BY SITE: Soil Organic Matter (%)")
print("="*70)
om_summary = trial_data.groupby(['site', 'treatment'])['organic_matter_pct'].agg(
    ['mean', 'std', 'count']
).round(3)
print(om_summary)

# Calculate treatment effect size (Cohen's d)
print("\n" + "="*70)
print("TREATMENT EFFECT SIZE BY SITE (Cohen's d)")
print("="*70)

for site in trial_data['site'].unique():
    site_data = trial_data[trial_data['site'] == site]
    
    notill = site_data[site_data['treatment'] == 'No-Till']['organic_matter_pct']
    conv = site_data[site_data['treatment'] == 'Conventional Tillage']['organic_matter_pct']
    
    # Cohen's d = (mean1 - mean2) / pooled_std
    pooled_std = np.sqrt((notill.std()**2 + conv.std()**2) / 2)
    cohens_d = (notill.mean() - conv.mean()) / pooled_std
    
    # Effect size interpretation
    if abs(cohens_d) < 0.2:
        interpretation = "negligible"
    elif abs(cohens_d) < 0.5:
        interpretation = "small"
    elif abs(cohens_d) < 0.8:
        interpretation = "medium"
    else:
        interpretation = "large"
    
    print(f"{site:20s}: d = {cohens_d:6.3f} ({interpretation} effect)")

## 5. Temporal Trends Analysis

Examining how no-till benefits change over time.

In [ ]:
# Visualize temporal trends
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Temporal Trends in Soil Health: No-Till Adoption Benefits Over Time',
             fontsize=16, fontweight='bold')

indicators_time = [
    ('organic_matter_pct', 'Soil Organic Matter (%)', axes[0,0]),
    ('aggregate_stability_pct', 'Aggregate Stability (%)', axes[0,1]),
    ('infiltration_cm_hr', 'Infiltration Rate (cm/hr)', axes[1,0]),
    ('crop_yield_bu_acre', 'Crop Yield (bu/acre)', axes[1,1])
]

for indicator, title, ax in indicators_time:
    # Calculate means by year, site, and treatment
    yearly_means = trial_data.groupby(['year', 'site', 'treatment'])[indicator].mean().reset_index()
    
    for site in trial_data['site'].unique():
        site_data = yearly_means[yearly_means['site'] == site]
        
        notill_data = site_data[site_data['treatment'] == 'No-Till']
        conv_data = site_data[site_data['treatment'] == 'Conventional Tillage']
        
        ax.plot(notill_data['year'], notill_data[indicator], 
               marker='o', linewidth=2, label=f'{site} - No-Till')
        ax.plot(conv_data['year'], conv_data[indicator], 
               marker='s', linewidth=2, linestyle='--', label=f'{site} - Conv', alpha=0.7)
    
    ax.set_xlabel('Year')
    ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'temporal_trends_notill.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Temporal trend analysis complete")

## 6. Multi-Indicator Soil Health Score

Creating composite soil health index for stakeholder communication.

In [ ]:
# Normalize indicators to 0-100 scale
def normalize_indicator(series, higher_is_better=True):
    """
    Normalize to 0-100 scale based on dataset range.
    """
    min_val = series.min()
    max_val = series.max()
    
    if higher_is_better:
        return ((series - min_val) / (max_val - min_val)) * 100
    else:
        return ((max_val - series) / (max_val - min_val)) * 100

# Create normalized scores
trial_data['om_score'] = normalize_indicator(trial_data['organic_matter_pct'], True)
trial_data['agg_score'] = normalize_indicator(trial_data['aggregate_stability_pct'], True)
trial_data['bd_score'] = normalize_indicator(trial_data['bulk_density_g_cm3'], False)  # Lower is better
trial_data['infil_score'] = normalize_indicator(trial_data['infiltration_cm_hr'], True)
trial_data['mbc_score'] = normalize_indicator(trial_data['microbial_biomass_C_mg_kg'], True)

# Composite soil health score (equal weighting)
trial_data['soil_health_score'] = (
    trial_data['om_score'] + 
    trial_data['agg_score'] + 
    trial_data['bd_score'] + 
    trial_data['infil_score'] + 
    trial_data['mbc_score']
) / 5

# Visualize composite scores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Composite Soil Health Score: Integrating Multiple Indicators',
             fontsize=16, fontweight='bold')

# Score by treatment and site
sns.boxplot(data=trial_data, x='site', y='soil_health_score', hue='treatment', ax=axes[0])
axes[0].set_title('Soil Health Score by Treatment', fontweight='bold')
axes[0].set_ylabel('Composite Soil Health Score (0-100)')
axes[0].set_xlabel('Site')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=15, ha='right')

# Score over time
yearly_scores = trial_data.groupby(['year', 'treatment'])['soil_health_score'].mean().reset_index()
for treatment in trial_data['treatment'].unique():
    treatment_data = yearly_scores[yearly_scores['treatment'] == treatment]
    marker = 'o' if treatment == 'No-Till' else 's'
    linestyle = '-' if treatment == 'No-Till' else '--'
    axes[1].plot(treatment_data['year'], treatment_data['soil_health_score'],
                marker=marker, linewidth=3, linestyle=linestyle, label=treatment)

axes[1].set_title('Soil Health Score Trends (Across All Sites)', fontweight='bold')
axes[1].set_ylabel('Composite Soil Health Score (0-100)')
axes[1].set_xlabel('Year')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'composite_soil_health_score.png'), dpi=150, bbox_inches='tight')
plt.show()

# Statistical test on composite score
perform_anova(trial_data, 'soil_health_score', 'Composite Soil Health Score')

## 7. Executive Summary for Stakeholders

Translating statistical findings into actionable recommendations.

In [ ]:
# Generate summary statistics for final report
summary_stats = trial_data.groupby(['treatment']).agg({
    'organic_matter_pct': ['mean', 'std'],
    'aggregate_stability_pct': ['mean', 'std'],
    'bulk_density_g_cm3': ['mean', 'std'],
    'infiltration_cm_hr': ['mean', 'std'],
    'crop_yield_bu_acre': ['mean', 'std'],
    'soil_health_score': ['mean', 'std']
}).round(2)

print("\n" + "="*80)
print("EXECUTIVE SUMMARY: NO-TILL ADOPTION TRIAL RESULTS")
print("="*80)

print("\n📊 KEY FINDINGS:")
print("\n1. SOIL HEALTH IMPROVEMENTS (Across all sites, 3-year average):")

notill_avg = trial_data[trial_data['treatment']=='No-Till']
conv_avg = trial_data[trial_data['treatment']=='Conventional Tillage']

improvements = {
    'Organic Matter': ((notill_avg['organic_matter_pct'].mean() - conv_avg['organic_matter_pct'].mean()) / 
                      conv_avg['organic_matter_pct'].mean() * 100),
    'Aggregate Stability': ((notill_avg['aggregate_stability_pct'].mean() - conv_avg['aggregate_stability_pct'].mean()) / 
                           conv_avg['aggregate_stability_pct'].mean() * 100),
    'Infiltration Rate': ((notill_avg['infiltration_cm_hr'].mean() - conv_avg['infiltration_cm_hr'].mean()) / 
                         conv_avg['infiltration_cm_hr'].mean() * 100),
    'Microbial Biomass': ((notill_avg['microbial_biomass_C_mg_kg'].mean() - conv_avg['microbial_biomass_C_mg_kg'].mean()) / 
                         conv_avg['microbial_biomass_C_mg_kg'].mean() * 100)
}

for metric, improvement in improvements.items():
    print(f"   • {metric}: +{improvement:.1f}% with no-till")

bd_improvement = ((conv_avg['bulk_density_g_cm3'].mean() - notill_avg['bulk_density_g_cm3'].mean()) / 
                 conv_avg['bulk_density_g_cm3'].mean() * 100)
print(f"   • Bulk Density: -{bd_improvement:.1f}% with no-till (lower is better)")

print("\n2. SITE-SPECIFIC RESPONSES:")
for site in trial_data['site'].unique():
    site_notill = trial_data[(trial_data['site']==site) & (trial_data['treatment']=='No-Till')]
    site_conv = trial_data[(trial_data['site']==site) & (trial_data['treatment']=='Conventional Tillage')]
    
    om_benefit = site_notill['organic_matter_pct'].mean() - site_conv['organic_matter_pct'].mean()
    print(f"   • {site}: +{om_benefit:.2f}% organic matter with no-till")

print("\n3. TEMPORAL TRENDS:")
year_2023 = trial_data[(trial_data['year']==2023) & (trial_data['treatment']=='No-Till')]['soil_health_score'].mean()
year_2025 = trial_data[(trial_data['year']==2025) & (trial_data['treatment']=='No-Till')]['soil_health_score'].mean()
improvement_over_time = ((year_2025 - year_2023) / year_2023) * 100

print(f"   • Soil health score improved {improvement_over_time:.1f}% from 2023 to 2025 under no-till")
print(f"   • Benefits compound over time - continued improvement expected")

print("\n4. ECONOMIC CONSIDERATIONS:")
yield_diff_2023 = trial_data[(trial_data['year']==2023) & (trial_data['treatment']=='No-Till')]['crop_yield_bu_acre'].mean() - \
                  trial_data[(trial_data['year']==2023) & (trial_data['treatment']=='Conventional Tillage')]['crop_yield_bu_acre'].mean()
yield_diff_2025 = trial_data[(trial_data['year']==2025) & (trial_data['treatment']=='No-Till')]['crop_yield_bu_acre'].mean() - \
                  trial_data[(trial_data['year']==2025) & (trial_data['treatment']=='Conventional Tillage')]['crop_yield_bu_acre'].mean()

print(f"   • Year 1 (2023): {yield_diff_2023:+.1f} bu/acre yield difference")
print(f"   • Year 3 (2025): {yield_diff_2025:+.1f} bu/acre yield difference")
print(f"   • Economic viability improves as system matures")

print("\n✅ RECOMMENDATIONS:")
print("   1. No-till adoption significantly improves soil health across diverse regions")
print("   2. Benefits increase over time - minimum 3-year commitment recommended")
print("   3. Greatest improvements in aggregate stability and microbial activity")
print("   4. Site-specific management needed to optimize results")
print("   5. Monitor soil health indicators annually to track progress")

print("\n" + "="*80)
print("Full statistical details available in ANOVA output above")
print("="*80)

## Summary

This analysis demonstrates:

✅ **Experimental design expertise** - Multi-site RCBD with proper blocking  
✅ **Statistical rigor** - ANOVA, post-hoc tests, effect size calculations  
✅ **Soil science knowledge** - Realistic treatment effects and agronomic interpretation  
✅ **Multi-site synthesis** - Identifying patterns across diverse environments  
✅ **Stakeholder communication** - Translating statistics into actionable recommendations  
✅ **Composite indicators** - Creating meaningful metrics for complex systems  
✅ **Publication-quality visualization** - Professional graphics for research dissemination  

**Skills directly applicable to Soil Health Institute:**
- Analyzing data from multi-site field trials
- Supporting statistical analysis for research publications
- Creating visualizations for stakeholder presentations
- Synthesizing findings across diverse soil types and climates
- Developing soil health assessment frameworks
- Communicating complex results to non-statisticians

In [ ]:
# Save processed data for database
trial_data.to_csv(os.path.join(output_dir, 'notill_trial_results_analyzed.csv'), index=False)
print("\n✅ Analysis complete. Results saved to notill_trial_results_analyzed.csv")